# M3 CLV 조건부 최초구매 카테고리 전이 그래프 - Dunnhumby

이 노트북은 **historical CLV proxy가 신규상품 후보의 방향을 바꾸는지** 빠르게 확인하는 seed 42 탐색 실험입니다.

- M1의 이진 사용자–상품 그래프와 LightGCN 전파는 그대로 유지합니다.
- CLV는 학습구간 장바구니 수 `N_hat`과 평균 장바구니 금액 `V_hat`의 곱 `N_hat × V_hat`으로 계산합니다.
- 연속 장바구니에서 `직전 상품 카테고리 → 다음 장바구니에서 처음 산 상품의 카테고리` 관계를 계산합니다.
- 같은 출발 카테고리라도 고객의 CLV 백분위에 따라 고CLV 방향과 저CLV 방향이 달라집니다.
- 고객 자신의 이력이 그 고객에게 적용될 관계를 만들지 않도록 사용자 5-fold 교차추정을 사용합니다.
- 최소 5명 지지도, pooled 전이로의 축소(`kappa=20`), 로그비 상한 `log(3)`, 고객당 상위 20개 목표 카테고리를 실행 전에 고정합니다.
- 일반 전이 대조군, 실제 CLV, 이진 구매 degree 10분위 내 CLV-shuffle을 같은 조건으로 학습합니다.
- 가격 입력, 외부 재정렬, 표본 가중, 추가 손실은 사용하지 않습니다.

DAY 1--683 학습, DAY 684--690 탐색 평가이며 validation 선택과 holdout은 없습니다. 주 판정은 실제 CLV의 Recall/NDCG @10·20·50 기하평균이 M1·일반 전이·CLV-shuffle을 모두 넘는지입니다. 한 개 seed 결과이므로 유의성이나 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = 'e85b6e44a3f9a92a3f648025b9abee0ca1d684d9'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_clv_conditioned_category_transition import (
    configure_clv_category_transition_run,
    preflight_summary,
    run_clv_category_transition_screen,
)

cfg = configure_clv_category_transition_run()
assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
summary = preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
assert summary['m3']['cross_fit_folds'] == 5
assert summary['m3']['minimum_distinct_user_support'] == 5
assert summary['m3']['max_target_categories_per_user'] == 20
assert summary['m3']['gamma'] == 0.075
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['reading_rule']['accuracy_guardrails'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_clv_category_transition_screen(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
]
available = [column for column in columns if column in result_df.columns]
display(result_df[available])

print('\n실제 CLV 귀속 판정:')
print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
print('\nCLV 조건부 그래프 진단:')
print(json.dumps(result_df.attrs['graph_diagnostics'], ensure_ascii=False, indent=2))
print('\n결과 파일:', result_df.attrs['result_paths'])